# Mistral Medical AI — GRPO Experiment 6 (Fixed & Optimized)

This notebook incorporates all critical fixes for post-SFT alignment using Group Relative Policy Optimization (GRPO):
1. **System Prompt Conditioning**: Injects `SYSTEM_PROMPT` into all prompts to prevent out-of-distribution drift and suppress ChatDoctor leakage.
2. **Strict Emergency Safety**: Immediate `-2.0` disqualification for unsafe delay/wait instructions.
3. **High-Contrast Boilerplate Reward**: `+1.0` bonus for direct answers; `-1.0` penalty for stock greetings.
4. **Conciseness & Natural EOS Reward**: Ensures answers terminate cleanly with punctuation instead of truncating at `max_tokens`.
5. **LoRA-Optimized GRPO Hyperparameters**: $K=6$ generations, `learning_rate=5e-5`, `beta=0.005`, `max_steps=100`.

**Required Kaggle Datasets:**
- `mistral-sft-adapter` (containing your trained SFT adapter)
- `medical-validated-conversations` (containing `integrated_conversations_ner_validated.csv`)
- `mistral-grpo-utils` (containing updated `grpo_utils.py`)


## Cell 1 — Check Environment
Verify CUDA, PyTorch, and installed library versions.

In [ ]:
import torch
import transformers
import bitsandbytes as bnb
import peft
import trl
import accelerate

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)

assert torch.cuda.is_available(), "GPU not detected! Select GPU T4 or P100 in Kaggle settings."
print("GPU Device:", torch.cuda.get_device_name(0))


## Cell 2 — Imports & Utility Module Discovery
Locates `grpo_utils.py` automatically and executes software self-tests.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel, prepare_model_for_kbit_training
from trl import GRPOConfig, GRPOTrainer

# Search for grpo_utils.py in /kaggle or local directory
matches = list(Path("/kaggle").rglob("grpo_utils.py"))
if not matches:
    matches = list(Path(".").rglob("grpo_utils.py"))

assert len(matches) >= 1, f"grpo_utils.py not found in /kaggle or working directory: {matches}"
UTILS_PATH = str(matches[0].parent)
if UTILS_PATH not in sys.path:
    sys.path.insert(0, UTILS_PATH)

import grpo_utils as gu

print("Utility file located at:", matches[0])
print("Utility module self-tests:", gu.run_utility_tests())
print("Active reward functions:", [f.__name__ for f in gu.make_reward_functions()])


In [ ]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# Locate CSV dataset
csv_matches = list(Path("/kaggle/input").rglob("integrated_conversations_ner_validated.csv"))
if not csv_matches:
    csv_matches = list(Path("data").rglob("integrated_conversations_ner_validated.csv"))
assert len(csv_matches) >= 1, f"Validated CSV not found: {csv_matches}"
CSV_PATH = str(csv_matches[0])

# Locate SFT Adapter checkpoint
adapter_configs = [
    p for p in Path("/kaggle/input").rglob("adapter_config.json")
    if "sft" in str(p).lower() or "mistral" in str(p).lower()
]
if not adapter_configs:
    adapter_configs = list(Path(".").rglob("adapter_config.json"))
assert len(adapter_configs) >= 1, f"SFT adapter config not found: {adapter_configs}"
ADAPTER_PATH = str(adapter_configs[0].parent)

OUT_DIR = "/kaggle/working/grpo_exp6"
os.makedirs(OUT_DIR, exist_ok=True)

print("Model ID:", MODEL_ID)
print("CSV Path:", CSV_PATH)
print("SFT Adapter Path:", ADAPTER_PATH)
print("Output Directory:", OUT_DIR)


## Cell 3 — Load and Filter Prompts
Extracts clean, non-emergency and emergency dialogue opening questions.

In [ ]:
df = pd.read_csv(
    CSV_PATH,
    usecols=["dialogue_id", "turn_id", "speaker", "utterance"],
    low_memory=False
)

df["utterance"] = df["utterance"].fillna("").astype(str).str.strip()
user_df = df[df["speaker"].astype(str).str.lower().eq("user")].copy()
user_df = user_df.sort_values(["dialogue_id", "turn_id"])
first_user = user_df.groupby("dialogue_id", sort=False).first().reset_index()

pool = first_user["utterance"].drop_duplicates()
pool = pool[pool.str.len().between(20, 350)]
pool = pool[~pool.str.contains(r"NAME_\d|^prompt:|^input:", case=False, regex=True)]
pool = pool[~pool.str.contains(
    r"suicid|kill myself|self[- ]?harm|end my life|want to die|\b(dog|cat|puppy|kitten|pet)\b",
    case=False, regex=True
)]

mined_df = pd.DataFrame({"question": pool.astype(str).tolist()})
mined_df["category"] = mined_df["question"].map(gu.mined_category)
mined_df = mined_df[mined_df["category"] != "skip"].drop_duplicates("question").reset_index(drop=True)

print("First user prompts available:", len(mined_df))
print("Category counts:")
print(mined_df["category"].value_counts(dropna=False).to_string())


## Cell 4 — Build Train/Eval Sets with System Prompt Conditioning
Injects the health assistant system prompt into every dialogue turn to keep inference in-distribution.

In [ ]:
fixed_prompts = [
    ("I have severe chest pain and difficulty breathing. What should I do?", "emergency"),
    ("Someone is unconscious and not responding. What should I do?", "emergency"),
    ("What should I do if someone may have taken an overdose?", "emergency"),
    ("My father's face is drooping and his speech is slurred. What should we do?", "emergency"),
    ("Can I change the dosage of my prescribed medication myself?", "medication"),
    ("What could be causing my frequent headaches?", "diagnosis"),
]

train_df, eval_df = gu.build_prompt_frames(
    mined_df=mined_df,
    fixed_prompts=fixed_prompts,
    seed=42,
    n_eval_mined=5,
    synthetic_train_n=60,
    synthetic_eval_n=16,
)

train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

# Construct TRL dataset with System Prompt conditioning
train_ds = Dataset.from_dict({
    "prompt": [gu.format_chat_prompt(q) for q in train_df["question"].astype(str)],
    "category": train_df["category"].astype(str).tolist(),
})

print(f"TRAIN PROMPTS: {len(train_df)}")
print(train_df["category"].value_counts().to_dict())
print(f"\nEVALUATION PROMPTS: {len(eval_df)}")
print(eval_df["category"].value_counts().to_dict())
print("\nExample formatted prompt for GRPOTrainer:")
for msg in train_ds[0]["prompt"]:
    print(f"  [{msg['role'].upper()}]: {msg['content'][:120]}...")


## Cell 5 — Load Tokenizer, Quantized Base Model, and SFT Adapter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=True,
)

for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

model.config.use_cache = False
model.print_trainable_parameters()
print("GPU Memory Allocated (GB):", round(torch.cuda.memory_allocated(0) / 1024**3, 2))


## Cell 6 — Generation Helper & Pre-Training SFT Baseline

In [ ]:
@torch.inference_mode()
def generate_answers(questions, batch_size=4, max_new_tokens=220):
    model.eval()
    old_cache = model.config.use_cache
    model.config.use_cache = True
    answers, ids_out = [], []
    try:
        for start in range(0, len(questions), batch_size):
            batch_questions = [str(q) for q in questions[start:start + batch_size]]
            prompts = [
                tokenizer.apply_chat_template(
                    gu.format_chat_prompt(q),
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for q in batch_questions
            ]
            encoded = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1024,
                add_special_tokens=False,
            ).to("cuda:0")
            outputs = model.generate(
                **encoded,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            generated = outputs[:, encoded["input_ids"].shape[1]:]
            for row in generated:
                ids = row.tolist()
                if tokenizer.eos_token_id in ids:
                    ids = ids[:ids.index(tokenizer.eos_token_id) + 1]
                ids_out.append(ids)
                answers.append(tokenizer.decode(ids, skip_special_tokens=True).strip())
            del encoded, outputs, generated
    finally:
        model.config.use_cache = old_cache
    return answers, ids_out

print("Generating SFT baseline evaluation answers...")
base_answers, base_ids = generate_answers(eval_df["question"].tolist())
base_metrics = gu.summarize_answers(
    eval_df, base_answers, base_ids, eos_token_id=tokenizer.eos_token_id
)

print("\nSFT Baseline Metrics (Conditioned on System Prompt):")
print(pd.Series(base_metrics).round(2).to_string())

pd.DataFrame({
    "category": eval_df["category"],
    "question": eval_df["question"],
    "sft_answer": base_answers,
}).to_csv(f"{OUT_DIR}/sft_baseline_answers.csv", index=False)


## Cell 7 — Reward Variance Probe
Verifies that each reward function produces non-zero variance when sampling $K=6$ completions.

In [ ]:
@torch.inference_mode()
def sample_group(question, n=6, max_new_tokens=220):
    model.eval()
    old_cache = model.config.use_cache
    model.config.use_cache = True
    try:
        prompt = tokenizer.apply_chat_template(
            gu.format_chat_prompt(question),
            tokenize=False,
            add_generation_prompt=True,
        )
        encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda:0")
        output = model.generate(
            **encoded,
            do_sample=True,
            temperature=0.85,
            top_p=0.95,
            num_return_sequences=n,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
        completions, ids_all = [], []
        for row in output[:, encoded["input_ids"].shape[1]:]:
            ids = row.tolist()
            if tokenizer.eos_token_id in ids:
                ids = ids[:ids.index(tokenizer.eos_token_id) + 1]
            ids_all.append(ids)
            completions.append(tokenizer.decode(ids, skip_special_tokens=True).strip())
        return completions, ids_all
    finally:
        model.config.use_cache = old_cache

probe_rows = train_df.groupby("category", sort=False).head(1).reset_index(drop=True)
probe_results = []
for _, row in probe_rows.iterrows():
    completions, comp_ids = sample_group(row["question"], n=6)
    cats = [row["category"]] * len(completions)
    for reward_name, reward_fn in gu.REWARDS.items():
        values = reward_fn(
            completions,
            category=cats,
            completion_ids=comp_ids,
            eos_token_id=tokenizer.eos_token_id,
        )
        applicable = [float(v) for v in values if v is not None]
        if len(applicable) > 1:
            probe_results.append({
                "category": row["category"],
                "reward": reward_name,
                "std": round(float(np.std(applicable)), 3),
                "scores": [round(x, 2) for x in applicable],
            })

probe_df = pd.DataFrame(probe_results)
print("Reward variance probe (N=6):")
print(probe_df.to_string(index=False))


## Cell 8 — Configure GRPO Hyperparameters
- $K = 6$ generations
- LoRA learning rate: `5e-5`
- KL penalty weight: `beta = 0.005`
- Max steps: `100`

In [ ]:
model.config.use_cache = False
torch.cuda.empty_cache()

args = GRPOConfig(
    output_dir=OUT_DIR,
    learning_rate=5e-5,
    warmup_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=6,
    num_generations=6,
    max_completion_length=220,
    max_steps=100,
    temperature=0.85,
    top_p=0.95,
    beta=0.005,
    reward_weights=[2.0, 1.2, 0.8, 0.5, 0.5],
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    gradient_checkpointing=True,
    logging_steps=1,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    seed=42,
)

args._n_gpu = 1

print("Trainer GPU count:", args._n_gpu)
print("Generations per prompt (K):", args.num_generations)
print("Learning rate:", args.learning_rate)
print("Reward weights:", args.reward_weights)
print("Training prompts:", len(train_ds))


## Cell 9 — Train with GRPOTrainer

In [ ]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=gu.make_reward_functions(),
    args=args,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

print("Starting GRPO training...")
train_result = trainer.train()
print("\nGRPO training completed successfully!")
print(train_result.metrics)

model = trainer.model
model.config.use_cache = False


## Cell 10 — Save Final Adapter and Inspect Logs

In [ ]:
FINAL_ADAPTER = f"{OUT_DIR}/final_adapter"
os.makedirs(FINAL_ADAPTER, exist_ok=True)
trainer.model.save_pretrained(FINAL_ADAPTER)
tokenizer.save_pretrained(FINAL_ADAPTER)

log_df = pd.DataFrame(trainer.state.log_history)
interesting = [
    col for col in log_df.columns
    if any(key in col.lower() for key in ["step", "reward", "zero_std", "clipped", "kl", "loss", "learning_rate"])
]
print("\nRecent Training Logs (Last 20 Steps):")
print(log_df[interesting].tail(20).to_string(index=False) if interesting else log_df.tail().to_string(index=False))

log_df.to_csv(f"{OUT_DIR}/training_log.csv", index=False)
print("\nSaved final adapter to:", FINAL_ADAPTER)


## Cell 11 — Evaluate SFT Baseline vs. GRPO Exp 6

In [ ]:
print("Generating evaluation answers with GRPO Exp 6 adapter...")
new_answers, new_ids = generate_answers(eval_df["question"].tolist())
new_metrics = gu.summarize_answers(
    eval_df, new_answers, new_ids, eos_token_id=tokenizer.eos_token_id
)

comparison = pd.DataFrame({
    "SFT baseline": base_metrics,
    "GRPO Exp 6": new_metrics,
    "Difference": {k: new_metrics[k] - base_metrics[k] for k in base_metrics}
}).round(2)

print("\n================== SFT vs GRPO EXP 6 COMPARISON ==================")
print(comparison.to_string())
comparison.to_csv(f"{OUT_DIR}/metrics_comparison.csv")

side_by_side = pd.DataFrame({
    "category": eval_df["category"],
    "question": eval_df["question"],
    "sft_answer": base_answers,
    "grpo_exp6_answer": new_answers,
})
side_by_side.to_csv(f"{OUT_DIR}/side_by_side.csv", index=False)

changed = sum(a != b for a, b in zip(base_answers, new_answers))
print(f"\nTotal answers changed: {changed}/{len(new_answers)}")

print("\n================== EMERGENCY COMPARISONS ==================")
for i, row in eval_df.iterrows():
    if row["category"] == "emergency_now":
        print(f"\nQ: {row['question'][:120]}")
        print("SFT :", gu.strip_greeting(base_answers[i])[:220])
        print("GRPO:", gu.strip_greeting(new_answers[i])[:220])


## Cell 12 — Zip Adapter for Download

In [ ]:
ZIP_PATH = shutil.make_archive(
    "/kaggle/working/grpo_exp6_adapter",
    "zip",
    FINAL_ADAPTER,
)

print("Adapter ZIP archive:", ZIP_PATH)
print("Archive size (MB):", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2))
print("Metrics comparison file:", f"{OUT_DIR}/metrics_comparison.csv")
print("Side-by-side CSV file:", f"{OUT_DIR}/side_by_side.csv")
